# Full Training Pipeline — SFT + DPO

Runs everything in one session on the **base** student model (`Llama-3.2-1B`, no built-in safety).

| Step | Task | Time |
|---|---|---|
| 1 | SFT Baseline (benign only) | ~1 hr |
| 2 | SFT With Refusals | ~1 hr |
| 3 | SFT Weighted | ~1 hr |
| 4 | Generate DPO pairs (base model → rejected responses) | ~30 min |
| 5 | DPO training on top of SFT baseline | ~1 hr |
| **Total** | | **~4.5 hrs** |

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add HuggingFace token: Add-ons → Secrets → **HF_TOKEN** (toggle On)

## Step 0: Install Dependencies

In [ ]:
!pip install -q transformers==4.46.3 datasets accelerate peft trl==0.12.2 huggingface_hub bitsandbytes tqdm

## Step 1: Setup

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("HuggingFace login successful.")

In [ ]:
REPO_URL = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
REPO_DIR = "/kaggle/working/Distill-Safety-Aligned-Models"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## Step 2: Configuration

In [ ]:
# ── Models ──────────────────────────────────────────────────────────────────
STUDENT_MODEL_ID = "meta-llama/Llama-3.2-1B"   # base model, NO safety training

# ── Data ────────────────────────────────────────────────────────────────────
TRAIN_FILE       = "data/train.jsonl"
EVAL_FILE        = "data/val.jsonl"
RAW_FILE         = "data/raw_teacher_outputs.jsonl"
DPO_PAIRS_FILE   = "data/dpo_pairs.jsonl"

# ── LoRA ────────────────────────────────────────────────────────────────────
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
TARGET_MODULES   = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# ── SFT hyperparams ──────────────────────────────────────────────────────────
MAX_LENGTH       = 1024
SFT_BATCH_SIZE   = 2    # reduced from 4 to avoid OOM
SFT_GRAD_ACCUM   = 16   # increased to keep effective batch = 32
SFT_EPOCHS       = 3
SFT_LR           = 2e-4
REFUSAL_WEIGHT   = 3.0

# ── DPO hyperparams ──────────────────────────────────────────────────────────
DPO_BATCH_SIZE   = 1    # DPO processes 4 sequences per pair; keep at 1 on T4
DPO_GRAD_ACCUM   = 32   # effective batch = 32
DPO_EPOCHS       = 2
DPO_LR           = 5e-5
DPO_BETA         = 0.1
MAX_PROMPT_LEN   = 512
DPO_MAX_LEN      = 1024
GEN_BATCH_SIZE   = 4    # reduced for safer inference
GEN_MAX_TOKENS   = 256

SEED = 42

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # better CUDA error messages
print(f"Student model: {STUDENT_MODEL_ID}")


## Step 3: Shared Utilities

In [ ]:
import json, re, torch
import torch.nn.functional as F
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, set_seed,
)
from trl import DPOConfig, DPOTrainer

set_seed(SEED)

SYSTEM_PROMPT = "You are a helpful, harmless, and honest assistant."

# ── Prompt formatting ────────────────────────────────────────────────────────
def build_prefix(prompt):
    return (
        "<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

def build_full(prompt, response):
    return f"{build_prefix(prompt)}{response}<|eot_id|>"

# ── I/O ─────────────────────────────────────────────────────────────────────
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def save_jsonl(records, path):
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ── Refusal detection ────────────────────────────────────────────────────────
REFUSAL_PREFIXES = [
    "i'm sorry", "i can't", "i cannot", "i'm not able", "i won't",
    "i will not", "as an ai", "i'm unable", "i apologize",
    "that's not something i", "i don't think i should",
    "this request involves", "i must decline",
]
REFUSAL_PATTERNS = [
    r"i (am|'m) (not able|unable) to",
    r"i (can't|cannot|won't|will not) (help|assist|provide|generate|create)",
    r"(this|that) (request|prompt|query) (is|seems|appears) (harmful|dangerous|unethical|illegal)",
]

def is_refusal(text):
    t = text.lower().strip()
    for p in REFUSAL_PREFIXES:
        if t.startswith(p):
            return True
    for pat in REFUSAL_PATTERNS:
        if re.search(pat, t):
            return True
    return False

print("Utilities loaded.")

In [ ]:
# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded.")

# ── SFT dataset encoding ─────────────────────────────────────────────────────
def encode_records(records, weight_fn):
    encoded, dropped = [], 0
    for r in records:
        prefix = build_prefix(r["prompt"])
        full   = build_full(r["prompt"], r["response"])
        prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
        enc = tokenizer(full, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
        ids, mask = enc["input_ids"], enc["attention_mask"]
        labels = list(ids)
        labels[:min(len(prefix_ids), len(ids))] = [-100] * min(len(prefix_ids), len(ids))
        if not any(l != -100 for l in labels):
            dropped += 1
            continue
        encoded.append({"input_ids": ids, "attention_mask": mask,
                        "labels": labels, "sample_weight": float(weight_fn(r))})
    print(f"  {len(encoded)} records encoded, {dropped} dropped")
    return Dataset.from_list(encoded)

def make_collator():
    pad_id = tokenizer.pad_token_id
    def collate(features):
        ml = max(len(f["input_ids"]) for f in features)
        ids, mask, labels, w = [], [], [], []
        for f in features:
            p = ml - len(f["input_ids"])
            ids.append(f["input_ids"] + [pad_id]*p)
            mask.append(f["attention_mask"] + [0]*p)
            labels.append(f["labels"] + [-100]*p)
            w.append(float(f["sample_weight"]))
        return {"input_ids": torch.tensor(ids, dtype=torch.long),
                "attention_mask": torch.tensor(mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "sample_weight": torch.tensor(w, dtype=torch.float)}
    return collate

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        w = inputs.pop("sample_weight", None)
        out = model(**inputs)
        sl = out.logits[:, :-1].contiguous()
        la = inputs["labels"][:, 1:].contiguous()
        tl = F.cross_entropy(sl.view(-1, sl.size(-1)), la.view(-1),
                             reduction="none", ignore_index=-100).view(la.size())
        mk = (la != -100).float()
        pl = (tl * mk).sum(1) / mk.sum(1).clamp_min(1.0)
        if w is None: w = torch.ones_like(pl)
        loss = (pl * w.to(pl.device)).sum() / w.sum().clamp_min(1e-8)
        return (loss, out) if return_outputs else loss

# ── Model loading helpers ────────────────────────────────────────────────────
def load_base_model():
    dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=dt,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_ID, trust_remote_code=True,
        quantization_config=bnb_config, device_map="auto")
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    return m, dt

def apply_lora(model):
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     bias="none", target_modules=TARGET_MODULES, task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

def sft_args(output_dir, dt):
    return TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        num_train_epochs=SFT_EPOCHS,
        learning_rate=SFT_LR,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=10, logging_first_step=True,
        save_strategy="epoch", save_total_limit=1,
        bf16=(dt==torch.bfloat16), fp16=(dt==torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none", remove_unused_columns=False,
        label_names=["labels"], seed=SEED,
    )

# ── Load data once ───────────────────────────────────────────────────────────
all_train = load_jsonl(TRAIN_FILE)
all_eval  = load_jsonl(EVAL_FILE)
benign_train    = [r for r in all_train if r.get("category") == "benign"]
benign_eval     = [r for r in all_eval  if r.get("category") == "benign"]
refusal_train   = [r for r in all_train if r.get("category") == "benign" or
                   (r.get("category") == "harmful" and r.get("is_refusal"))]
refusal_eval    = [r for r in all_eval  if r.get("category") == "benign" or
                   (r.get("category") == "harmful" and r.get("is_refusal"))]
print(f"Benign train: {len(benign_train)}, With refusals train: {len(refusal_train)}")

## Step 4: SFT Variant 1 — Baseline (benign only)
Never sees refusals → expect **low** safety refusal rate.

⏱ ~1 hour

In [ ]:
print("=== SFT BASELINE ===")
train_ds = encode_records(benign_train,  lambda r: 1.0)
eval_ds  = encode_records(benign_eval,   lambda r: 1.0)

model, dt = load_base_model()
model = apply_lora(model)
trainer = WeightedTrainer(model=model, args=sft_args("outputs/baseline", dt),
    train_dataset=train_ds, eval_dataset=eval_ds, data_collator=make_collator())
result = trainer.train()
trainer.save_model("outputs/baseline")
tokenizer.save_pretrained("outputs/baseline")
print("Baseline done:", result.metrics)
del model, trainer; torch.cuda.empty_cache()

## Step 5: SFT Variant 2 — With Refusals
Includes teacher refusals → model explicitly learns to refuse.

⏱ ~1 hour

In [ ]:
print("=== SFT WITH REFUSALS ===")
train_ds = encode_records(refusal_train, lambda r: 1.0)
eval_ds  = encode_records(refusal_eval,  lambda r: 1.0)

model, dt = load_base_model()
model = apply_lora(model)
trainer = WeightedTrainer(model=model, args=sft_args("outputs/with_refusals", dt),
    train_dataset=train_ds, eval_dataset=eval_ds, data_collator=make_collator())
result = trainer.train()
trainer.save_model("outputs/with_refusals")
tokenizer.save_pretrained("outputs/with_refusals")
print("With refusals done:", result.metrics)
del model, trainer; torch.cuda.empty_cache()

## Step 6: SFT Variant 3 — Weighted
Refusal examples count 3× more in the loss.

⏱ ~1 hour

In [ ]:
print("=== SFT WEIGHTED ===")
def w_fn(r):
    return REFUSAL_WEIGHT if (r.get("category")=="harmful" and r.get("is_refusal")) else 1.0

train_ds = encode_records(refusal_train, w_fn)
eval_ds  = encode_records(refusal_eval,  w_fn)

model, dt = load_base_model()
model = apply_lora(model)
trainer = WeightedTrainer(model=model, args=sft_args("outputs/weighted", dt),
    train_dataset=train_ds, eval_dataset=eval_ds, data_collator=make_collator())
result = trainer.train()
trainer.save_model("outputs/weighted")
tokenizer.save_pretrained("outputs/weighted")
print("Weighted done:", result.metrics)
del model, trainer; torch.cuda.empty_cache()

## Step 7: Generate DPO Preference Pairs

Run the raw **base model** on harmful prompts to get rejected (compliant) responses.
Pair with teacher refusals as chosen.

⏱ ~30 min

In [ ]:
print("=== GENERATING DPO PAIRS ===")
raw_records = load_jsonl(RAW_FILE)
harmful_refusals = [r for r in raw_records
                    if r.get("category")=="harmful" and r.get("is_refusal", False)]
print(f"Harmful refusal records: {len(harmful_refusals)}")

# Load bare base model (no LoRA) — guaranteed to comply with harmful prompts
tokenizer.padding_side = "left"
dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
gen_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID, trust_remote_code=True, torch_dtype=dt, device_map="auto")
gen_model.eval()

prompts = [r["prompt"]   for r in harmful_refusals]
chosen  = [r["response"] for r in harmful_refusals]
rejected = []

for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc="generating rejected"):
    batch = prompts[i : i + GEN_BATCH_SIZE]
    chats = [[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":p}] for p in batch]
    inputs_text = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in chats]
    inputs = tokenizer(inputs_text, return_tensors="pt", padding=True,
                       truncation=True, max_length=1024).to(gen_model.device)
    with torch.no_grad():
        outputs = gen_model.generate(**inputs, max_new_tokens=GEN_MAX_TOKENS,
                                     do_sample=False, pad_token_id=tokenizer.pad_token_id,
                                     eos_token_id=tokenizer.eos_token_id)
    input_len = inputs["input_ids"].shape[1]
    for out in outputs:
        rejected.append(tokenizer.decode(out[input_len:], skip_special_tokens=True).strip())

pairs, skipped = [], 0
for p, ch, rej in zip(prompts, chosen, rejected):
    if is_refusal(rej):
        skipped += 1
        continue
    pairs.append({"prompt": p, "chosen": ch, "rejected": rej})

print(f"Valid DPO pairs: {len(pairs)}, Skipped: {skipped}")
save_jsonl(pairs, DPO_PAIRS_FILE)
print(f"Saved → {DPO_PAIRS_FILE}")

del gen_model; torch.cuda.empty_cache()
tokenizer.padding_side = "right"

## Step 8: DPO Training

Builds on top of the SFT baseline. Merges baseline LoRA → adds fresh DPO LoRA → trains with DPO loss.

⏱ ~1 hour

In [ ]:
print("=== DPO TRAINING ===")
tokenizer.padding_side = "left"

# Build dataset in TRL messages format
dpo_pairs = load_jsonl(DPO_PAIRS_FILE)
dpo_dataset = Dataset.from_list([{
    "prompt":   [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":p["prompt"]}],
    "chosen":   [{"role":"assistant","content":p["chosen"]}],
    "rejected": [{"role":"assistant","content":p["rejected"]}],
} for p in dpo_pairs])
print(f"DPO dataset: {len(dpo_dataset)} pairs")

# Load base model, merge SFT baseline LoRA, attach fresh DPO LoRA
dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID, trust_remote_code=True, torch_dtype=dt, device_map="auto")
sft = PeftModel.from_pretrained(base, "outputs/baseline")
merged = sft.merge_and_unload()
print("SFT baseline merged.")

lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                      bias="none", target_modules=TARGET_MODULES, task_type="CAUSAL_LM")
policy = get_peft_model(merged, lora_cfg)
policy.config.use_cache = False
policy.print_trainable_parameters()

dpo_config = DPOConfig(
    output_dir="outputs/dpo",
    beta=DPO_BETA,
    loss_type="sigmoid",
    max_prompt_length=MAX_PROMPT_LEN,
    max_length=DPO_MAX_LEN,
    per_device_train_batch_size=DPO_BATCH_SIZE,
    gradient_accumulation_steps=DPO_GRAD_ACCUM,
    num_train_epochs=DPO_EPOCHS,
    learning_rate=DPO_LR,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    logging_steps=10, logging_first_step=True,
    save_strategy="epoch", save_total_limit=1,
    eval_strategy="no",
    bf16=(dt==torch.bfloat16), fp16=(dt==torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none", remove_unused_columns=False, seed=SEED,
)

trainer = DPOTrainer(model=policy, ref_model=None, args=dpo_config,
                     train_dataset=dpo_dataset, processing_class=tokenizer)
result = trainer.train()
trainer.save_model("outputs/dpo")
tokenizer.save_pretrained("outputs/dpo")
print("DPO done:", result.metrics)

summary = {"num_pairs": len(dpo_pairs), "beta": DPO_BETA,
           "learning_rate": DPO_LR, "epochs": DPO_EPOCHS, "metrics": result.metrics}
with open("outputs/dpo/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## Step 9: Download Outputs

Zips all adapters + DPO pairs for download from the Output tab.

In [ ]:
import shutil
base_dir = "/kaggle/working/Distill-Safety-Aligned-Models"

shutil.make_archive("/kaggle/working/sft_baseline",      "zip", base_dir, "outputs/baseline")
shutil.make_archive("/kaggle/working/sft_with_refusals", "zip", base_dir, "outputs/with_refusals")
shutil.make_archive("/kaggle/working/sft_weighted",      "zip", base_dir, "outputs/weighted")
shutil.make_archive("/kaggle/working/dpo",               "zip", base_dir, "outputs/dpo")
shutil.copy(f"{base_dir}/data/dpo_pairs.jsonl", "/kaggle/working/dpo_pairs.jsonl")

print("Done! Download from the Output tab:")
print("  sft_baseline.zip, sft_with_refusals.zip, sft_weighted.zip, dpo.zip, dpo_pairs.jsonl")